## Example: Compositional Design of Biological Circuits using Formal Specifications

### Ayush Pandey
### Date: April 2, 2026

In this example, we consider a compositional design problem of a cascade of three subsystems: $\Sigma_1$, $\Sigma_2$, and $\Sigma_3$. The composed system $\Sigma$ is a composition of all three subsystems. 

### Goals: 

* Validate whether the given subsystems are comaptible for composition.
* Verify that the composed system $\Sigma$ satisfies a given top-level desired specification.

In [8]:
import math
import pandas as pd
from pacti.contracts import PolyhedralIoContract
import numpy as np

In [17]:
# functions to create contracts based on parameter sets for different designs
def create_contract_Ey(theta1):
    # enzyme-saturated contract
    K = theta1["K"]
    kcat = theta1["kcat"]
    delta_ux = theta1["delta_ux"]
    delta_uK = theta1["delta_uK"]

    zK = np.log(K)
    zkcat = np.log(kcat)

    eta_01 = -np.log((1.0 - np.exp(-delta_ux)) / (1.0 + np.exp(-delta_uK)))

    contract_Ey_local = PolyhedralIoContract.from_dict({
        "input_vars": ["z_u1", "z_x1"],
        "output_vars": ["z_y1"],
        "assumptions": [
            # z_u - z_x >= delta_ux
            {"coefficients": {"z_x1": 1.0, "z_u1": -1.0}, "constant": -delta_ux},
            # z_u - zK >= delta_uK
            {"coefficients": {"z_u1": -1.0}, "constant": -(zK + delta_uK)},
        ],
        "guarantees": [
            # z_y <= z_x + zkcat
            {"coefficients": {"z_y1": 1.0, "z_x1": -1.0}, "constant": zkcat},
            # z_y >= z_x + zkcat - eta_01
            {"coefficients": {"z_x1": 1.0, "z_y1": -1.0}, "constant": eta_01 - zkcat},
        ],
    })
    return contract_Ey_local


def create_contract_Sy(theta2):
    # substrate-saturated contract

    K = theta2["K"]
    kcat = theta2["kcat"]
    delta_xu = theta2["delta_xu"]
    delta_xK = theta2["delta_xK"]

    zK = np.log(K)
    zkcat = np.log(kcat)

    eta_10 = -np.log((1.0 - np.exp(-delta_xu)) / (1.0 + np.exp(-delta_xK)))

    contract_Sy_local = PolyhedralIoContract.from_dict({
        "input_vars": ["z_u2", "z_x2"],
        "output_vars": ["z_y2"],
        "assumptions": [
            # z_x - z_u >= delta_xu
            {"coefficients": {"z_u2": 1.0, "z_x2": -1.0}, "constant": -delta_xu},
            # z_x - zK >= delta_xK
            {"coefficients": {"z_x2": -1.0}, "constant": -(zK + delta_xK)},
        ],
        "guarantees": [
            # z_y <= z_u + zkcat
            {"coefficients": {"z_y2": 1.0, "z_u2": -1.0}, "constant": zkcat},
            # z_y >= z_u + zkcat - eta_10
            {"coefficients": {"z_u2": 1.0, "z_y2": -1.0}, "constant": eta_10 - zkcat},
        ],
    })
    return contract_Sy_local



def create_contract_C3(theta3):
    K31 = theta3["K31"]
    K32 = theta3["K32"]
    kcat31 = theta3["kcat31"]
    kcat32 = theta3["kcat32"]
    delta_mm = theta3["delta_mm"]
    zx_lo = theta3["zx_lo"]
    zx_hi = theta3["zx_hi"]

    zK31 = np.log(K31)
    zK32 = np.log(K32)
    zkcat31 = np.log(kcat31)
    zkcat32 = np.log(kcat32)

    def phi_mm(zx, zK):
        return -np.log(1.0 + np.exp(zK - zx))

    def dphi_mm(zx, zK):
        return 1.0 / (1.0 + np.exp(zx - zK))

    # branch 31
    phi_lo_31 = phi_mm(zx_lo, zK31)
    phi_hi_31 = phi_mm(zx_hi, zK31)
    m_sec_31 = (phi_hi_31 - phi_lo_31) / (zx_hi - zx_lo)
    b_sec_31 = phi_lo_31 - m_sec_31 * zx_lo
    m_tan_lo_31 = dphi_mm(zx_lo, zK31)
    b_tan_lo_31 = phi_lo_31 - m_tan_lo_31 * zx_lo
    m_tan_hi_31 = dphi_mm(zx_hi, zK31)
    b_tan_hi_31 = phi_hi_31 - m_tan_hi_31 * zx_hi

    # branch 32
    phi_lo_32 = phi_mm(zx_lo, zK32)
    phi_hi_32 = phi_mm(zx_hi, zK32)
    m_sec_32 = (phi_hi_32 - phi_lo_32) / (zx_hi - zx_lo)
    b_sec_32 = phi_lo_32 - m_sec_32 * zx_lo
    m_tan_lo_32 = dphi_mm(zx_lo, zK32)
    b_tan_lo_32 = phi_lo_32 - m_tan_lo_32 * zx_lo
    m_tan_hi_32 = dphi_mm(zx_hi, zK32)
    b_tan_hi_32 = phi_hi_32 - m_tan_hi_32 * zx_hi

    contract_C3_local = PolyhedralIoContract.from_dict({
        "input_vars": ["z_y1", "z_y2", "z_x3"],
        "output_vars": ["z_y31", "z_y32"],
        "assumptions": [
            # shared x-window
            {"coefficients": {"z_x3": -1.0}, "constant": -zx_lo},
            {"coefficients": {"z_x3":  1.0}, "constant":  zx_hi},

            # MM assumption for branch 31
            {"coefficients": {"z_y1": 1.0, "z_x3": -1.0}, "constant": -delta_mm},

            # MM assumption for branch 32
            {"coefficients": {"z_y2": 1.0, "z_x3": -1.0}, "constant": -delta_mm},
        ],
        "guarantees": [
            # lower secant bound for y31
            {"coefficients": {"z_y1": 1.0, "z_x3": m_sec_31, "z_y31": -1.0},
                "constant": -(zkcat31 + b_sec_31)},
            # upper tangent bounds for y31
            {"coefficients": {"z_y31": 1.0, "z_y1": -1.0, "z_x3": -m_tan_lo_31},
                "constant": zkcat31 + b_tan_lo_31},
            {"coefficients": {"z_y31": 1.0, "z_y1": -1.0, "z_x3": -m_tan_hi_31},
                "constant": zkcat31 + b_tan_hi_31},

            # lower secant bound for y32
            {"coefficients": {"z_y2": 1.0, "z_x3": m_sec_32, "z_y32": -1.0},
                "constant": -(zkcat32 + b_sec_32)},
            # upper tangent bounds for y32
            {"coefficients": {"z_y32": 1.0, "z_y2": -1.0, "z_x3": -m_tan_lo_32},
                "constant": zkcat32 + b_tan_lo_32},
            {"coefficients": {"z_y32": 1.0, "z_y2": -1.0, "z_x3": -m_tan_hi_32},
                "constant": zkcat32 + b_tan_hi_32},
        ],
    })

    return contract_C3_local

# printing for getting results out
def fmt_theta1(t):
    return rf"$K_1={t['K']:.3g},\;k_{{cat,1}}={t['kcat']:.3g},\;\delta_{{ux,1}}={t['delta_ux']:.3g},\;\delta_{{uK,1}}={t['delta_uK']:.3g}$"

def fmt_theta2(t):
    return rf"$K_2={t['K']:.3g},\;k_{{cat,2}}={t['kcat']:.3g},\;\delta_{{xu,2}}={t['delta_xu']:.3g},\;\delta_{{xK,2}}={t['delta_xK']:.3g}$"

def fmt_theta3(t):
    return rf"$K_{{31}}={t['K31']:.3g},\;K_{{32}}={t['K32']:.3g},\;k_{{cat,31}}={t['kcat31']:.3g},\;k_{{cat,32}}={t['kcat32']:.3g}$"




Composition and refinement diagnosis methods


In [18]:

def interval_overlap(lo1, hi1, lo2, hi2, tol=1e-9):
    if lo1 is None or hi1 is None or lo2 is None or hi2 is None:
        return True
    return max(lo1, lo2) <= min(hi1, hi2) + tol

def get_bounds_safe(C, var_name):
    try:
        return C.get_variable_bounds(var_name)
    except Exception:
        return (None, None)

def diagnose_compose_failure(C1, C2, C3):
    # Heuristic: check whether C1/C2 output ranges overlap the C3 admissible input ranges
    y1_lo, y1_hi = get_bounds_safe(C1, "z_y1")
    y2_lo, y2_hi = get_bounds_safe(C2, "z_y2")
    u31_lo, u31_hi = get_bounds_safe(C3, "z_y1")
    u32_lo, u32_hi = get_bounds_safe(C3, "z_y2")

    bad1 = not interval_overlap(y1_lo, y1_hi, u31_lo, u31_hi)
    bad2 = not interval_overlap(y2_lo, y2_hi, u32_lo, u32_hi)

    if bad1 and bad2:
        return r"$\Sigma_1,\Sigma_2$"
    if bad1:
        return r"$\Sigma_1$"
    if bad2:
        return r"$\Sigma_2$"
    return r"$\Sigma_3$"
def diagnose_refinement_failure(C1, C2, C3, Cfull, Cspec):
    culprit = diagnose_compose_failure(C1, C2, C3)
    if culprit != r"$\Sigma_3$":
        return culprit

    y31_lo_full, y31_hi_full = get_bounds_safe(Cfull, "z_y31")
    y31_lo_spec, y31_hi_spec = get_bounds_safe(Cspec, "z_y31")

    y32_lo_full, y32_hi_full = get_bounds_safe(Cfull, "z_y32")
    y32_lo_spec, y32_hi_spec = get_bounds_safe(Cspec, "z_y32")

    bad31 = False
    bad32 = False

    if y31_hi_full is not None and y31_lo_spec is not None and y31_hi_full < y31_lo_spec:
        bad31 = True
    if y31_lo_full is not None and y31_hi_spec is not None and y31_lo_full > y31_hi_spec:
        bad31 = True

    if y32_hi_full is not None and y32_lo_spec is not None and y32_hi_full < y32_lo_spec:
        bad32 = True
    if y32_lo_full is not None and y32_hi_spec is not None and y32_lo_full > y32_hi_spec:
        bad32 = True

    if bad31 and bad32:
        return r"$\Sigma_3$"
    if bad31:
        return r"$\Sigma_3$ (branch 31)"
    if bad32:
        return r"$\Sigma_3$ (branch 32)"

    return r"mixed / not isolated"




Composition of contracts with Pacti

In [19]:
def compose_design(C1, C2, C3):
    C12 = C1.compose(C2, vars_to_keep=["z_y1", "z_y2"])
    Cfull = C12.compose(C3, vars_to_keep=["z_y31", "z_y32"])
    return C12, Cfull


Example designs, each with different parameter sets:

In [20]:

designs = [
    {
        "name": "D1",
        "theta1": {"K": 1.0, "kcat": 1.0, "delta_ux": 0.7, "delta_uK": 3.7},
        "theta2": {"K": 1.0, "kcat": 1.0, "delta_xu": 1.2, "delta_xK": 1.6},
        "theta3": {"K31": 1.0, "K32": 1.0, "kcat31": 1.0, "kcat32": 1.0, "zx_lo": math.log(15.0), "zx_hi": math.log(40.0), "delta_mm": 2.3},
    },
    {
        "name": "D2",
        "theta1": {"K": 1.0, "kcat": 1.1, "delta_ux": 0.9, "delta_uK": 4.0},
        "theta2": {"K": 1.0, "kcat": 0.9, "delta_xu": 1.4, "delta_xK": 1.7},
        "theta3": {"K31": 1.1, "K32": 1.0, "kcat31": 1.0, "kcat32": 1.0, "zx_lo": math.log(15.0), "zx_hi": math.log(40.0), "delta_mm": 2.5},
    },
    {
        "name": "D3",
        "theta1": {"K": 1.0, "kcat": 0.9, "delta_ux": 0.6, "delta_uK": 3.4},
        "theta2": {"K": 1.0, "kcat": 1.2, "delta_xu": 1.0, "delta_xK": 1.3},
        "theta3": {"K31": 0.9, "K32": 1.1, "kcat31": 1.1, "kcat32": 0.9, "zx_lo": math.log(15.0), "zx_hi": math.log(40.0), "delta_mm": 2.0},
    },
    {
        "name": "D4",
        "theta1": {"K": 1.0, "kcat": 1.4, "delta_ux": 1.1, "delta_uK": 4.4},
        "theta2": {"K": 1.0, "kcat": 0.8, "delta_xu": 1.6, "delta_xK": 1.9},
        "theta3": {"K31": 1.0, "K32": 1.2, "kcat31": 1.2, "kcat32": 0.8, "zx_lo": math.log(15.0), "zx_hi": math.log(40.0), "delta_mm": 2.7},
    },
    {
        "name": "D5",
        "theta1": {"K": 1.0, "kcat": 0.8, "delta_ux": 0.5, "delta_uK": 3.0},
        "theta2": {"K": 1.0, "kcat": 1.3, "delta_xu": 0.9, "delta_xK": 1.2},
        "theta3": {"K31": 1.2, "K32": 0.9, "kcat31": 0.9, "kcat32": 1.2, "zx_lo": math.log(15.0), "zx_hi": math.log(40.0), "delta_mm": 1.9},
    },
]

In [22]:
# the line below artifically creates an anchor around D5 
# to demonstrate the results. Specifically, this lets us run a sanity 
# check that D5 design must meet top-level specs.
anchor_design = next(d for d in designs if d["name"] == "D5")


C1_anchor = create_contract_Ey(anchor_design["theta1"])
C2_anchor = create_contract_Sy(anchor_design["theta2"])
C3_anchor = create_contract_C3(anchor_design["theta3"])
_, Cfull_anchor = compose_design(C1_anchor, C2_anchor, C3_anchor)

md = Cfull_anchor.to_machine_dict()

# use pacti to compute safe bounds and add small slack so the spec is not too strict
y31_lo_anchor, y31_hi_anchor = get_bounds_safe(Cfull_anchor, "z_y31")
y32_lo_anchor, y32_hi_anchor = get_bounds_safe(Cfull_anchor, "z_y32")

if y31_hi_anchor is None:
    y31_hi_spec = 10.0
else:
    y31_hi_spec = float(y31_hi_anchor) + 0.35

if y32_hi_anchor is None:
    y32_hi_spec = 10.0
else:
    y32_hi_spec = float(y32_hi_anchor) + 0.35

spec_dict = {
    "input_vars": md["input_vars"],
    "output_vars": md["output_vars"],
    "assumptions": md["assumptions"],
    "guarantees": [
        # upper-only guarantees
        {"coefficients": {"z_y31": 1.0}, "constant": y31_hi_spec},
        {"coefficients": {"z_y32": 1.0}, "constant": y32_hi_spec},
    ],
}

Cspec = PolyhedralIoContract.from_dict(spec_dict)

# sanity check on the anchor
assumptions_check = Cspec.a <= Cfull_anchor.a
guarantees_check = (Cfull_anchor.g | Cspec.a) <= (Cspec.g | Cspec.a)
print("assumptions pass: ", assumptions_check)
print("guarantees pass: ", guarantees_check)
print("True if refinement succeeds \n (since this is trivial sanity check it should print True) \n=", Cfull_anchor.refines(Cspec))

assumptions pass:  True
guarantees pass:  True
True if refinement succeeds 
 (since this is trivial sanity check it should print True) 
= True


Now check all designs for compositionality and refinement to the top-level specification:

In [23]:

rows = []

for d in designs:
    C1 = create_contract_Ey(d["theta1"])
    C2 = create_contract_Sy(d["theta2"])
    C3 = create_contract_C3(d["theta3"])

    compose_ok = True
    refine_ok = False
    culprit = ""

    try:
        C12, Cfull = compose_design(C1, C2, C3)
    except Exception as e:
        compose_ok = False
        refine_ok = False
        culprit = "composition failed"
        rows.append({
            "Design": d["name"],
            r"$\Sigma_1$ params": fmt_theta1(d["theta1"]),
            r"$\Sigma_2$ params": fmt_theta2(d["theta2"]),
            r"$\Sigma_3$ params": fmt_theta3(d["theta3"]),
            r"$C_{\mathrm{full}} \preceq C_{\mathrm{spec}}$": "No (compose failed)",
            "Failed subsystem": culprit,
        })
        print(f"Design {d['name']}: compose_ok=False, refine_ok=False, culprit={culprit}")
        continue

    assumptions_check = Cspec.a <= Cfull.a
    guarantees_check = (Cfull.g | Cspec.a) <= (Cspec.g | Cspec.a)
    refine_ok = assumptions_check and guarantees_check

    if refine_ok:
        culprit = "--"
    elif not assumptions_check:
        culprit = "assumptions"
    else:
        culprit = "guarantees"

    rows.append({
        "Design": d["name"],
        r"$\Sigma_1$ params": fmt_theta1(d["theta1"]),
        r"$\Sigma_2$ params": fmt_theta2(d["theta2"]),
        r"$\Sigma_3$ params": fmt_theta3(d["theta3"]),
        r"$C_{\mathrm{full}} \preceq C_{\mathrm{spec}}$": "Yes" if refine_ok else "No",
        "Failed subsystem": culprit,
    })

    print(
        f"Design {d['name']}: "
        f"compose_ok={compose_ok}, "
        f"assumptions_check={assumptions_check}, "
        f"guarantees_check={guarantees_check}, "
        f"refine_ok={refine_ok}, "
        f"culprit={culprit}"
    )


Design D1: compose_ok=True, assumptions_check=False, guarantees_check=True, refine_ok=False, culprit=assumptions
Design D2: compose_ok=True, assumptions_check=False, guarantees_check=False, refine_ok=False, culprit=assumptions
Design D3: compose_ok=True, assumptions_check=False, guarantees_check=True, refine_ok=False, culprit=assumptions
Design D4: compose_ok=True, assumptions_check=False, guarantees_check=False, refine_ok=False, culprit=assumptions
Design D5: compose_ok=True, assumptions_check=True, guarantees_check=True, refine_ok=True, culprit=--


D1, D3, and D5 were reported in the paper.

Write report in LaTeX for the paper:

In [24]:

selected_names = ["D1", "D3", "D5"]

def fmt_design_compact(d):
    t1 = d["theta1"]
    t2 = d["theta2"]
    t3 = d["theta3"]
    return (
        f"{d['name']}: "
        f"(K1:{t1['K']}, "
        f"K2:{t2['K']}, "
        f"K31:{t3['K31']}, K32:{t3['K32']})"
    )
rows = []

for d in designs:
    if d["name"] not in selected_names:
        continue

    C1 = create_contract_Ey(d["theta1"])
    C2 = create_contract_Sy(d["theta2"])
    C3 = create_contract_C3(d["theta3"])

    try:
        _, Cfull = compose_design(C1, C2, C3)
        compose_ok = True
    except Exception:
        compose_ok = False

    if compose_ok:
        assumptions_check = (Cspec.a <= Cfull.a)
        guarantees_check = ((Cfull.g | Cspec.a) <= (Cspec.g | Cspec.a))
        refine_ok = assumptions_check and guarantees_check
    else:
        assumptions_check = False
        guarantees_check = False
        refine_ok = False

    rows.append({
        "Design": fmt_design_compact(d),
        "Composition": "Yes" if compose_ok else "No",
        r"$A_{\mathrm{spec}} \subseteq A_{\mathrm{full}}$": "Yes" if assumptions_check else "No",
        r"$\left(G_{\mathrm{full}}\mid A_{\mathrm{spec}}\right)\subseteq\left(G_{\mathrm{spec}}\mid A_{\mathrm{spec}}\right)$": "Yes" if guarantees_check else "No",
        r"$C_{\mathrm{full}} \preceq C_{\mathrm{spec}}$": "Yes" if refine_ok else "No",
    })

df = pd.DataFrame(rows)[[
    "Design",
    "Composition",
    r"$A_{\mathrm{spec}} \subseteq A_{\mathrm{full}}$",
    r"$\left(G_{\mathrm{full}}\mid A_{\mathrm{spec}}\right)\subseteq\left(G_{\mathrm{spec}}\mid A_{\mathrm{spec}}\right)$",
    r"$C_{\mathrm{full}} \preceq C_{\mathrm{spec}}$",
]]

latex_table = df.to_latex(
    index=False,
    escape=False,
    column_format="p{8.5cm} c c c c",
)

print(latex_table)

\begin{tabular}{p{8.5cm} c c c c}
\toprule
Design & Composition & $A_{\mathrm{spec}} \subseteq A_{\mathrm{full}}$ & $\left(G_{\mathrm{full}}\mid A_{\mathrm{spec}}\right)\subseteq\left(G_{\mathrm{spec}}\mid A_{\mathrm{spec}}\right)$ & $C_{\mathrm{full}} \preceq C_{\mathrm{spec}}$ \\
\midrule
D1: (K1:1.0, K2:1.0, K31:1.0, K32:1.0) & Yes & No & Yes & No \\
D3: (K1:1.0, K2:1.0, K31:0.9, K32:1.1) & Yes & No & Yes & No \\
D5: (K1:1.0, K2:1.0, K31:1.2, K32:0.9) & Yes & Yes & Yes & Yes \\
\bottomrule
\end{tabular}

